In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

TABLA_BRONZE = "proyecto_final.bronze.ventas"
TABLA_SILVER = "proyecto_final.silver.ventas"
TABLA_CUARENTENA = "proyecto_final.silver.cuarentena"

df_bronze = spark.table(TABLA_BRONZE)

print("=== ESQUEMA BRONZE VENTAS ===")
df_bronze.printSchema()

print("\n=== TOTAL REGISTROS ===")
print(df_bronze.count())

print("\n=== MUESTRA DE VENTAS ===")
display(df_bronze.limit(10))

In [0]:
from pyspark.sql import functions as F

print("=== DISTRIBUCIÓN POR ARCHIVO ===")

display(
    df_bronze
    .groupBy("archivo_origen")
    .agg(
        F.count("*").alias("total_registros"),
        F.countDistinct("venta_id").alias("ventas_distintas"),
        F.min("fecha_ingesta").alias("primera_ingesta"),
        F.max("fecha_ingesta").alias("ultima_ingesta")
    )
    .orderBy("archivo_origen")
)

print("=== VALIDACIÓN INICIAL ===")

display(
    df_bronze.agg(
        F.count("*").alias("total_bronze"),

        F.countDistinct("venta_id")
            .alias("ventas_distintas"),

        F.sum(
            (
                F.col("venta_id").isNull() |
                (F.trim("venta_id") == "")
            ).cast("int")
        ).alias("venta_id_vacio"),

        F.sum(
            (
                F.col("sucursal_id").isNull() |
                (F.trim("sucursal_id") == "")
            ).cast("int")
        ).alias("sucursal_vacia"),

        F.sum(
            (
                F.col("producto_id").isNull() |
                (F.trim("producto_id") == "")
            ).cast("int")
        ).alias("producto_vacio"),

        F.sum(
            (
                F.col("monto_total").isNull() |
                (F.trim(F.col("monto_total").cast("string")) == "")
            ).cast("int")
        ).alias("monto_vacio"),

        F.sum(
            F.col("vendedor_id").isNull().cast("int")
        ).alias("vendedor_nulo")
    )
)

In [0]:
from pyspark.sql import functions as F

df_ventas_analisis = (
    df_bronze
    .withColumn(
        "monto_texto",
        F.trim(F.col("monto_total"))
    )
    .withColumn(
        "monto_normalizado",
        F.when(
            F.col("monto_texto").rlike(
                r"^-?\d{1,3}(\.\d{3})+,\d{1,2}$"
            ),
            F.regexp_replace(
                F.regexp_replace("monto_texto", r"\.", ""),
                ",", "."
            )
        ).otherwise(
            F.regexp_replace("monto_texto", ",", ".")
        )
    )
    .withColumn(
        "monto_decimal",
        F.expr(
            "try_cast(monto_normalizado AS DECIMAL(12,2))"
        )
    )
    .withColumn(
        "cantidad_int",
        F.expr("try_cast(trim(cantidad) AS INT)")
    )
    .withColumn(
        "fecha_venta_ts",
        F.coalesce(
            F.try_to_timestamp(
                "fecha_venta", F.lit("yyyy-MM-dd")
            ),
            F.try_to_timestamp(
                "fecha_venta", F.lit("dd/MM/yyyy")
            ),
            F.try_to_timestamp(
                "fecha_venta", F.lit("yyyy-MM-dd HH:mm:ss")
            ),
            F.try_to_timestamp(
                "fecha_venta", F.lit("yyyy-MM-dd'T'HH:mm:ss")
            )
        )
    )
)

print("=== VALIDACIONES CRÍTICAS ===")

display(
    df_ventas_analisis.agg(
        F.count("*").alias("total_bronze"),

        F.sum(
            F.col("monto_decimal").isNull().cast("int")
        ).alias("montos_nulos_o_invalidos"),

        F.sum(
            (F.col("monto_decimal") <= 0).cast("int")
        ).alias("montos_no_positivos"),

        F.sum(
            (
                F.col("cantidad_int").isNull() |
                (F.col("cantidad_int") <= 0)
            ).cast("int")
        ).alias("cantidades_invalidas"),

        F.sum(
            F.col("fecha_venta_ts").isNull().cast("int")
        ).alias("fechas_invalidas"),

        F.sum(
            F.col("vendedor_id").isNull().cast("int")
        ).alias("vendedores_nulos")
    )
)

print("=== MÉTODOS DE PAGO ===")

display(
    df_ventas_analisis
    .groupBy("metodo_pago")
    .count()
    .orderBy(F.desc("count"))
)

print("=== CANALES ===")

display(
    df_ventas_analisis
    .groupBy("canal")
    .count()
    .orderBy(F.desc("count"))
)

print("=== VENDEDORES NULOS POR CANAL ===")

display(
    df_ventas_analisis
    .groupBy("canal")
    .agg(
        F.count("*").alias("total_ventas"),
        F.sum(
            F.col("vendedor_id").isNull().cast("int")
        ).alias("vendedores_nulos")
    )
    .orderBy("canal")
)

In [0]:
from pyspark.sql import functions as F

# 1. Normalizar los 17 métodos de pago
df_ventas_normalizadas = (
    df_ventas_analisis
    .withColumn(
        "pago_limpio",
        F.lower(F.trim("metodo_pago"))
    )
    .withColumn(
        "metodo_pago_normalizado",
        F.when(
            F.col("pago_limpio").isin(
                "tarjeta", "tarjeta de credito", "tc", "td"
            ), "tarjeta"
        ).when(
            F.col("pago_limpio").isin("efectivo", "efv"),
            "efectivo"
        ).when(
            F.col("pago_limpio").isin(
                "transferencia", "trf"
            ), "transferencia"
        ).when(
            F.col("pago_limpio") == "yape", "yape"
        ).when(
            F.col("pago_limpio") == "plin", "plin"
        )
    )
)

print("=== METODOS DE PAGO NORMALIZADOS ===")

display(
    df_ventas_normalizadas
    .groupBy("metodo_pago_normalizado")
    .count()
    .orderBy("metodo_pago_normalizado")
)

# 2. Comprobar si los 240 duplicados son idénticos
columnas_negocio = [
    "sucursal_id", "producto_id", "vendedor_id",
    "cantidad", "monto_total", "metodo_pago",
    "fecha_venta", "canal"
]

df_duplicados_ventas = (
    df_ventas_normalizadas
    .withColumn(
        "hash_contenido",
        F.sha2(
            F.to_json(
                F.struct(*[F.col(c) for c in columnas_negocio])
            ),
            256
        )
    )
    .groupBy("venta_id")
    .agg(
        F.count("*").alias("apariciones"),
        F.countDistinct("hash_contenido")
            .alias("versiones_contenido")
    )
    .filter(F.col("apariciones") > 1)
)

print("=== VALIDACION DE DUPLICADOS ===")

display(
    df_duplicados_ventas.agg(
        F.count("*").alias("ventas_repetidas"),
        F.sum(F.col("apariciones") - 1)
            .alias("filas_adicionales"),
        F.sum(
            (F.col("versiones_contenido") > 1).cast("int")
        ).alias("ventas_con_cambios"),
        F.sum(
            (F.col("versiones_contenido") == 1).cast("int")
        ).alias("ventas_identicas")
    )
)

# 3. Referencias existentes en Bronze y Silver Catálogo
df_ids_catalogo = (
    spark.table("proyecto_final.bronze.catalogo")
    .select("producto_id")
    .unionByName(
        spark.table("proyecto_final.silver.catalogo")
        .select("producto_id")
    )
    .distinct()
    .withColumn("producto_existe", F.lit(True))
)

# 4. Referencias a las sucursales ya validadas
df_ids_sucursales = (
    spark.table("proyecto_final.silver.sucursales")
    .select("sucursal_id")
    .distinct()
    .withColumn("sucursal_existe", F.lit(True))
)

df_referencias_ventas = (
    df_ventas_normalizadas
    .join(df_ids_catalogo, "producto_id", "left")
    .join(df_ids_sucursales, "sucursal_id", "left")
)

print("=== REFERENCIAS Y PAGOS ===")

display(
    df_referencias_ventas.agg(
        F.count("*").alias("total_ventas"),
        F.sum(
            F.col("producto_existe").isNull().cast("int")
        ).alias("productos_huerfanos"),
        F.sum(
            (
                F.col("sucursal_id").isNotNull()
                & (F.trim("sucursal_id") != "")
                & F.col("sucursal_existe").isNull()
            ).cast("int")
        ).alias("sucursales_inexistentes"),
        F.sum(
            F.col("metodo_pago_normalizado").isNull()
            .cast("int")
        ).alias("pagos_no_normalizados")
    )
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Conservar una aparición por venta_id
# Los duplicados de negocio ya se comprobaron idénticos.

w_venta = (
    Window.partitionBy("venta_id")
    .orderBy(
        F.col("fecha_ingesta").desc_nulls_last(),
        F.col("archivo_origen").desc_nulls_last()
    )
)

df_ventas_unicas = (
    df_referencias_ventas
    .withColumn("rn", F.row_number().over(w_venta))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# 2. Aplicar reglas críticas
df_calidad_ventas = (
    df_ventas_unicas

    .withColumn(
        "rechazo_monto",
        F.col("monto_decimal").isNull() |
        (F.col("monto_decimal") <= 0)
    )

    .withColumn(
        "rechazo_cantidad",
        F.col("cantidad_int").isNull() |
        (F.col("cantidad_int") <= 0)
    )

    .withColumn(
        "rechazo_sucursal",
        F.col("sucursal_id").isNull() |
        (F.trim("sucursal_id") == "") |
        F.col("sucursal_existe").isNull()
    )

    .withColumn(
        "rechazo_producto",
        F.col("producto_id").isNull() |
        (F.trim("producto_id") == "") |
        F.col("producto_existe").isNull()
    )

    .withColumn(
        "rechazo_fecha",
        F.col("fecha_venta_ts").isNull()
    )

    .withColumn(
        "rechazo_id",
        F.col("venta_id").isNull() |
        (F.trim("venta_id") == "")
    )

    .withColumn(
        "rechazo_pago",
        F.col("metodo_pago_normalizado").isNull()
    )

    .withColumn(
        "rechazo_canal",
        ~F.coalesce(
            F.col("canal").isin("tienda_fisica", "ecommerce"),
            F.lit(False)
        )
    )

    .withColumn(
        "rechazado",
        F.col("rechazo_monto") |
        F.col("rechazo_cantidad") |
        F.col("rechazo_sucursal") |
        F.col("rechazo_producto") |
        F.col("rechazo_fecha") |
        F.col("rechazo_id") |
        F.col("rechazo_pago") |
        F.col("rechazo_canal")
    )

    .withColumn(
        "motivo_rechazo",
        F.concat_ws(
            ", ",
            F.when(F.col("rechazo_monto"), "MONTO_INVALIDO"),
            F.when(F.col("rechazo_cantidad"), "CANTIDAD_INVALIDA"),
            F.when(F.col("rechazo_sucursal"), "SUCURSAL_INVALIDA"),
            F.when(F.col("rechazo_producto"), "PRODUCTO_HUERFANO"),
            F.when(F.col("rechazo_fecha"), "FECHA_INVALIDA"),
            F.when(F.col("rechazo_id"), "VENTA_ID_INVALIDO"),
            F.when(F.col("rechazo_pago"), "PAGO_INVALIDO"),
            F.when(F.col("rechazo_canal"), "CANAL_INVALIDO")
        )
    )
)

# 3. Conciliación
print("=== CONCILIACIÓN DE CALIDAD VENTAS ===")

display(
    df_calidad_ventas.agg(
        F.count("*").alias("ventas_unicas"),

        F.sum(F.col("rechazo_monto").cast("int"))
            .alias("montos_invalidos"),

        F.sum(F.col("rechazo_cantidad").cast("int"))
            .alias("cantidades_invalidas"),

        F.sum(F.col("rechazo_sucursal").cast("int"))
            .alias("sucursales_invalidas"),

        F.sum(F.col("rechazo_producto").cast("int"))
            .alias("productos_huerfanos"),

        F.sum(F.col("rechazo_fecha").cast("int"))
            .alias("fechas_invalidas"),

        F.sum(F.col("rechazo_pago").cast("int"))
            .alias("pagos_invalidos"),

        F.sum(F.col("rechazo_canal").cast("int"))
            .alias("canales_invalidos"),

        F.sum(F.col("rechazado").cast("int"))
            .alias("total_cuarentena"),

        F.sum((~F.col("rechazado")).cast("int"))
            .alias("total_silver")
    )
)

print("=== RECHAZOS AGRUPADOS POR MOTIVO ===")

display(
    df_calidad_ventas
    .filter(F.col("rechazado"))
    .groupBy("motivo_rechazo")
    .count()
    .orderBy(F.desc("count"))
)

print("=== VENTAS ACEPTADAS POR CANAL ===")

display(
    df_calidad_ventas
    .filter(~F.col("rechazado"))
    .groupBy("canal")
    .agg(
        F.count("*").alias("ventas_aceptadas"),
        F.sum("monto_decimal").alias("monto_total")
    )
    .orderBy("canal")
)

In [0]:
from pyspark.sql import functions as F

df_silver_ventas = (
    df_calidad_ventas
    .filter(~F.col("rechazado"))
    .select(
        F.trim(F.col("venta_id")).alias("venta_id"),
        F.trim(F.col("sucursal_id")).alias("sucursal_id"),
        F.trim(F.col("producto_id")).alias("producto_id"),
        F.when(
            F.col("vendedor_id").isNull() |
            (F.trim(F.col("vendedor_id")) == ""),
            F.lit(None).cast("string")
        ).otherwise(
            F.trim(F.col("vendedor_id"))
        ).alias("vendedor_id"),
        F.col("cantidad_int").cast("int").alias("cantidad"),
        F.col("monto_decimal")
            .cast("decimal(12,2)").alias("monto_total"),
        F.col("metodo_pago_normalizado").alias("metodo_pago"),
        F.col("fecha_venta_ts").cast("timestamp").alias("fecha_venta"),
        F.col("canal")
    )
)

print("=== ESQUEMA FINAL SILVER VENTAS ===")
df_silver_ventas.printSchema()

print("=== VALIDACIÓN PREVIA A SILVER ===")

display(
    df_silver_ventas.agg(
        F.count("*").alias("total_ventas"),
        F.countDistinct("venta_id").alias("ventas_distintas"),
        F.sum("monto_total").alias("monto_total"),
        F.sum(
            F.col("monto_total").isNull().cast("int")
        ).alias("montos_nulos"),
        F.sum(
            (F.col("cantidad") <= 0).cast("int")
        ).alias("cantidades_invalidas")
    )
)

assert df_silver_ventas.count() == 17420, \
    "El total de Silver no coincide con la conciliación."

print("Preparación Silver correcta.")

In [0]:
TABLA_SILVER = "proyecto_final.silver.ventas"

(
    df_silver_ventas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_SILVER)
)

print("Tabla Silver Ventas creada correctamente.")

In [0]:
%sql
SELECT
    venta_id,
    sucursal_id,
    producto_id,
    vendedor_id,
    cantidad,
    monto_total,
    metodo_pago,
    fecha_venta,
    canal
FROM proyecto_final.silver.ventas
ORDER BY venta_id
LIMIT 30;

In [0]:
from pyspark.sql import functions as F

TABLA_CUARENTENA = "proyecto_final.silver.cuarentena"

# Conservar exclusivamente las columnas originales de Bronze
columnas_bronze = df_bronze.columns

df_cuarentena_ventas = (
    df_calidad_ventas
    .filter(F.col("rechazado"))
    .select(
        F.lit("ventas").alias("fuente"),

        F.col("venta_id").alias("clave_registro"),

        F.col("motivo_rechazo"),

        F.current_timestamp().alias("fecha_rechazo"),

        F.to_json(
            F.struct(
                *[F.col(c) for c in columnas_bronze]
            )
        ).alias("datos_originales"),

        F.col("archivo_origen")
    )
)

print("=== REGISTROS PREPARADOS PARA CUARENTENA ===")

display(
    df_cuarentena_ventas
    .groupBy("motivo_rechazo")
    .count()
    .orderBy(F.desc("count"))
)

print("=== VALIDACIÓN DE CUARENTENA ===")

display(
    df_cuarentena_ventas.agg(
        F.count("*").alias("total_rechazados"),
        F.countDistinct("clave_registro")
            .alias("ventas_distintas_rechazadas"),
        F.sum(
            F.col("datos_originales").isNull().cast("int")
        ).alias("registros_sin_datos_originales")
    )
)

# Verificar conciliación
total_rechazados = df_cuarentena_ventas.count()
total_silver = df_silver_ventas.count()

assert total_rechazados == 580, \
    f"Se esperaban 580 rechazos, se encontraron {total_rechazados}"

assert total_silver + total_rechazados + 240 == 18240, \
    "La conciliación Bronze, Silver y cuarentena no coincide."

print("Conciliación correcta:")
print(f"Bronze: 18240")
print(f"Duplicados eliminados: 240")
print(f"Silver: {total_silver}")
print(f"Cuarentena preparada: {total_rechazados}")

In [0]:
from pyspark.sql import functions as F

TABLA_CUARENTENA = "proyecto_final.silver.cuarentena"

# Verificar que la preparación sea correcta
assert df_cuarentena_ventas.count() == 580, \
    "El total de rechazos no coincide con la conciliación."

# Eliminar únicamente los rechazos anteriores de Ventas.
# Esto permite ejecutar nuevamente la carga sin duplicarlos.
spark.sql(f"""
    DELETE FROM {TABLA_CUARENTENA}
    WHERE fuente = 'ventas'
""")

# Insertar los 580 registros preparados
(
    df_cuarentena_ventas
    .select(
        "fuente",
        "clave_registro",
        "motivo_rechazo",
        "fecha_rechazo",
        "datos_originales",
        "archivo_origen"
    )
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(TABLA_CUARENTENA)
)

print("=== CARGA DE CUARENTENA VENTAS FINALIZADA ===")

display(
    spark.table(TABLA_CUARENTENA)
    .filter(F.col("fuente") == "ventas")
    .groupBy("motivo_rechazo")
    .count()
    .orderBy(F.desc("count"))
)

total_cargado = (
    spark.table(TABLA_CUARENTENA)
    .filter(F.col("fuente") == "ventas")
    .count()
)

assert total_cargado == 580, \
    f"Se esperaban 580 rechazos, se encontraron {total_cargado}"

print(f"Registros de Ventas cargados en cuarentena: {total_cargado}")

In [0]:
%sql
SELECT
    fuente,
    clave_registro,
    motivo_rechazo,
    fecha_rechazo,
    datos_originales,
    archivo_origen
FROM proyecto_final.silver.cuarentena
WHERE fuente = 'ventas'
ORDER BY clave_registro
LIMIT 50;